In [4]:
import re
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Dict, Optional, Tuple, Union, List

NS = {
    "g": "http://graphml.graphdrawing.org/xmlns",
    "y": "http://www.yworks.com/xml/graphml",
}
_LABEL_RE = re.compile(r"^\s*(-?\d+)\s*,\s*(-?\d+)\s*$")

def _ensure_rectangle_shape(shape_el):
    """Ensure the node uses a rectangle shape in yEd."""
    shp = shape_el.find("./y:Shape", NS)
    if shp is None:
        shp = ET.SubElement(shape_el, f"{{{NS['y']}}}Shape")
    shp.set("type", "rectangle")

def snap_nodes_to_grid_by_label(
    graphml_in: Union[str, Path],
    graphml_out: Union[str, Path],
    *,
    cell: Optional[float] = 50.0,         # FORCE a 50px grid by default
    margin_left: float = 40.0,
    margin_top: float = 40.0,
    node_size: float = 12.0,              # small square side length in px
    center_labels: bool = True,
) -> None:
    """
    Snap nodes labeled as 'row,col' to a strict square grid and render them as small squares.
    - Grid spacing is exactly `cell` pixels (default 50).
    - Node size is a fixed square `node_size` (default 12 px).
    - Works with y:ShapeNode (recommended). If a y:Shape child exists, it's set to rectangle.
    """
    tree = ET.parse(str(graphml_in))
    root = tree.getroot()

    nodes: Dict[str, Dict] = {}
    min_row = +10**9
    max_row = -10**9
    min_col = +10**9
    max_col = -10**9

    # Collect every node
    for n in root.findall(".//g:graph/g:node", NS):
        nid = n.attrib.get("id")
        if not nid:
            continue

        # Find yEd visual payload (prefer ShapeNode)
        shape = None
        for d in n.findall("./g:data", NS):
            cand = d.find("./y:ShapeNode", NS)
            if cand is not None:
                shape = cand
                break
            # fallback: GenericNode (we'll still move geometry)
            cand = d.find("./y:GenericNode", NS)
            if cand is not None:
                shape = cand
                break
        if shape is None:
            continue

        geom = shape.find("./y:Geometry", NS)
        if geom is None:
            continue

        x = float(geom.attrib.get("x", "0"))
        y = float(geom.attrib.get("y", "0"))
        w = float(geom.attrib.get("width", "0"))
        h = float(geom.attrib.get("height", "0"))

        label_el = shape.find("./y:NodeLabel", NS)
        label_txt = label_el.text.strip() if (label_el is not None and label_el.text) else None

        row = col = None
        if label_txt:
            m = _LABEL_RE.match(label_txt)
            if m:
                row = int(m.group(1))
                col = int(m.group(2))

        if row is None or col is None:
            continue

        nodes[nid] = {
            "elem": n,
            "shape": shape,
            "geom": geom,
            "label": label_el,
            "row": row, "col": col,
            "x": x, "y": y, "w": w, "h": h,
        }

        min_row = min(min_row, row)
        max_row = max(max_row, row)
        min_col = min(min_col, col)
        max_col = max(max_col, col)

    if not nodes:
        raise RuntimeError("No nodes with parseable labels 'row,col' were found.")

    # Enforce exact grid spacing
    if cell is None:
        cell = 50.0

    # Fixed small square size
    forced_w = float(node_size)
    forced_h = float(node_size)

    # Reposition and restyle
    for _, info in nodes.items():
        row, col = info["row"], info["col"]

        cx = margin_left + (col - min_col) * cell + (cell / 2.0)
        cy = margin_top  + (row - min_row) * cell + (cell / 2.0)

        w = forced_w
        h = forced_h
        x = cx - w / 2.0
        y = cy - h / 2.0

        geom = info["geom"]
        geom.set("x", f"{x:.6f}")
        geom.set("y", f"{y:.6f}")
        geom.set("width",  f"{w:.6f}")
        geom.set("height", f"{h:.6f}")

        # Make shape a rectangle (square)
        _ensure_rectangle_shape(info["shape"])

        # (Optional) center label visually
        if center_labels and info["label"] is not None:
            lbl = info["label"]
            # Best-effort center; yEd will finalize on open
            lbl.set("alignment", "center")
            lbl.set("horizontalTextPosition", "center")
            lbl.set("verticalTextPosition", "bottom")
            # If label has width/height attrs, attempt local centering offsets
            lw = float(lbl.attrib.get("width", "0") or "0")
            lh = float(lbl.attrib.get("height", "0") or "0")
            if lw > 0:
                lbl.set("x", f"{(w - lw) / 2.0:.6f}")
            if lh > 0:
                lbl.set("y", f"{(h - lh) / 2.0:.6f}")

    ET.register_namespace("", NS["g"])
    ET.register_namespace("y", NS["y"])
    ET.ElementTree(root).write(str(graphml_out), encoding="utf-8", xml_declaration=True)


In [9]:
snap_nodes_to_grid_by_label(
    graphml_in="../mapGraphs/graphml/HIMCM_graph_FINAL.graphml",
    graphml_out="../mapGraphs/graphml/HIMCM_graph_cleaned.graphml",
    cell=49,          # exact 50px spacing
    node_size=12.0,     # small square size; tweak as you like
    margin_left=40.0,
    margin_top=40.0,
    center_labels=True,
)